# Lesson 25: Feature engineering · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 25

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Fee per weekly hour

Create a fee_per_hour column (monthly_fee_usd divided by avg_weekly_hours). Store its median for subscribers who churned (churned = 1) in med_churn, and for those who stayed (churned = 0) in med_stay.

<details><summary>Hint</summary>

Divide one column by the other. medians is a Series indexed by 0 and 1, so medians[1] picks the churned group.

</details>

In [ ]:
import pandas as pd

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})

subs["fee_per_hour"] = ...
medians = subs.groupby("churned")["fee_per_hour"].median()
med_churn = ...
med_stay = ...
print(med_churn, med_stay)

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_s[\"fph\"] = _s[\"monthly_fee_usd\"] / _s[\"avg_weekly_hours\"]\nassert \"fee_per_hour\" in subs.columns and ((subs[\"fee_per_hour\"] - _s[\"fph\"]).abs() < 1e-9).all(), \"fee_per_hour should be monthly_fee_usd divided by avg_weekly_hours\"\n_m = _s.groupby(\"churned\")[\"fph\"].median()\nassert med_churn is not Ellipsis and abs(float(med_churn) - _m[1]) < 1e-9, \"med_churn should be the median for churned subscribers: medians[1]\"\nassert med_stay is not Ellipsis and abs(float(med_stay) - _m[0]) < 1e-9, \"med_stay should be the median for subscribers who stayed: medians[0]\"\n")

## Exercise 2: Bands and one-hot columns

Make a tenure_band column with pd.cut(subs["tenure_months"], bins=[0, 6, 12, 24, 72], labels=["0-6", "7-12", "13-24", "25+"]) and store the churn rate of the "0-6" band in rate_new. Then store one-hot columns for plan in plan_cols with pd.get_dummies(subs["plan"], prefix="plan", dtype=int).

<details><summary>Hint</summary>

pd.cut(...) makes the bands; groupby("tenure_band", observed=True)["churned"].mean() gives a rate per band, and ["0-6"] picks one. pd.get_dummies does the one-hot encoding.

</details>

In [ ]:
import pandas as pd

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})

subs["tenure_band"] = ...
rate_new = ...
plan_cols = ...
print(rate_new)
print(plan_cols.head())

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\n_want = _s.loc[_s[\"tenure_months\"] <= 6, \"churned\"].mean()\nassert rate_new is not Ellipsis and abs(float(rate_new) - _want) < 1e-9, \"rate_new should be the churn rate of the 0-6 band: groupby(\\\"tenure_band\\\", observed=True)[\\\"churned\\\"].mean()[\\\"0-6\\\"]\"\nassert isinstance(plan_cols, _pd.DataFrame), \"plan_cols should be a DataFrame from pd.get_dummies\"\nassert sorted(plan_cols.columns) == [\"plan_Basic\", \"plan_Premium\", \"plan_Standard\"], \"Use pd.get_dummies(subs[\\\"plan\\\"], prefix=\\\"plan\\\", dtype=int)\"\nassert (plan_cols.sum(axis=1) == 1).all() and int(plan_cols[\"plan_Basic\"].sum()) == int((_s[\"plan\"] == \"Basic\").sum()), \"Each row should have exactly one 1 across the plan columns\"\n")

## Exercise 3: Does it help the tree?

Add fee_per_hour and tickets_per_month (support_tickets_90d ÷ tenure_months). Split as in the lesson, train a DecisionTreeClassifier(max_depth=3, random_state=42) on the base columns plus the two ratios, and store the number of test churners it catches (predicted 1 and really 1) in caught.

<details><summary>Hint</summary>

tree.fit(train[cols], train["churned"]), then pred = tree.predict(test[cols]). Combine two conditions with &.

</details>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

subs = pd.read_csv(DATA_URL.replace("loans.csv", "global/subscribers.csv"))
subs["avg_weekly_hours"] = subs["avg_weekly_hours"].fillna(subs["avg_weekly_hours"].median())
subs["monthly_contract"] = subs["contract"].map({"Annual": 0, "Monthly": 1})
subs["fee_per_hour"] = ...
subs["tickets_per_month"] = ...

base = ["age", "monthly_fee_usd", "tenure_months", "avg_weekly_hours",
        "devices", "support_tickets_90d", "auto_renew", "monthly_contract"]
cols = base + ["fee_per_hour", "tickets_per_month"]
train, test = train_test_split(subs, test_size=0.2, random_state=42, stratify=subs["churned"])

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
# fit on train[cols], predict test[cols]
pred = ...
caught = ...
print(caught)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_s = subs.copy() if \"subs\" in globals() and hasattr(subs, \"copy\") else None\nimport pandas as _pd\n_s = _pd.read_csv(DATA_URL.replace(\"loans.csv\", \"global/subscribers.csv\"))\n_s[\"avg_weekly_hours\"] = _s[\"avg_weekly_hours\"].fillna(_s[\"avg_weekly_hours\"].median())\n_s[\"monthly_contract\"] = _s[\"contract\"].map({\"Annual\": 0, \"Monthly\": 1})\n_base = [\"age\", \"monthly_fee_usd\", \"tenure_months\", \"avg_weekly_hours\", \"devices\", \"support_tickets_90d\", \"auto_renew\", \"monthly_contract\"]\nfrom sklearn.model_selection import train_test_split as _tts\nfrom sklearn.tree import DecisionTreeClassifier as _DT\n_s[\"fee_per_hour\"] = _s[\"monthly_fee_usd\"] / _s[\"avg_weekly_hours\"]\n_s[\"tickets_per_month\"] = _s[\"support_tickets_90d\"] / _s[\"tenure_months\"]\nassert \"tickets_per_month\" in subs.columns and ((subs[\"tickets_per_month\"] - _s[\"tickets_per_month\"]).abs() < 1e-9).all(), \"tickets_per_month should be support_tickets_90d / tenure_months\"\nassert \"fee_per_hour\" in subs.columns and ((subs[\"fee_per_hour\"] - _s[\"fee_per_hour\"]).abs() < 1e-9).all(), \"fee_per_hour should be monthly_fee_usd / avg_weekly_hours\"\n_c = _base + [\"fee_per_hour\", \"tickets_per_month\"]\n_tr, _te = _tts(_s, test_size=0.2, random_state=42, stratify=_s[\"churned\"])\n_p = _DT(max_depth=3, random_state=42).fit(_tr[_c], _tr[\"churned\"]).predict(_te[_c])\nassert caught is not Ellipsis and int(caught) == int(((_p == 1) & (_te[\"churned\"] == 1)).sum()), \"caught should count test subscribers predicted 1 who really churned: ((pred == 1) & (test[\\\"churned\\\"] == 1)).sum()\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=25) and take the quiz.